# 第12回　情報カスケードと集団意思決定
## ―― 個々は合理的なのに、集団そろって間違える

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

いよいよ今期のクライマックスに入る。これまで「独立が崩れると統計の道具が壊れる」を繰り返し見てきた。今回は、人間の集団で **独立が壊れる現場** を目撃する。一人ひとりは賢く合理的なのに、互いを真似た結果、全員そろって間違える ―― **情報カスケード** だ。▶ を上から押そう。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
print("準備OK。次のセルへ。")

---
## 1. 直感クイズ ―― 行列のできる店

2軒のラーメン店があり、本当はどちらが美味しいかは分からない。通りかかった人が1人ずつ、「どちらに入るか」を決めていく。

各人は ―― ①自分なりの **私的な勘**（口コミを少し聞いた、外観を見た等。完璧ではないが当たる確率は五分より上）と、②**先に並んでいる人数**（他人の選択）を見て決める。

**問い：あなたが店Aに並ぶと、次に来た人には「Aに○人並んだ店」に見える。これが続くと、集団はどうなる？**　予想を決めてから ▶。

---
## 2. モデル ―― 私的な情報 vs 他人の行動

古典的な「壺と玉」のモデルで考える。

- 正解（本当に良い選択）は **1** だとする（学生には知らされない設定）。
- 各人は **私的シグナル** を1つ受け取る。これは確率 **70%** で正解を指す（五分よりは賢いが、完璧ではない）。
- 各人は、自分より **前の人たちの“選択”** も見える（ただし、その人の私的シグナルは見えない）。
- 各人はベイズ的に **合理的に** 判断する：公開された選択の流れと自分のシグナルを総合して、良さそうな方を選ぶ。

合理的な判断ルールはこう要約できる ―― **「先行者の選択が一方に2つ以上傾いていたら、自分のシグナル1つではもう覆せないので、自分のシグナルを捨てて多数に従う」**。

---
## 3. 運の悪い出だしが、全員を道連れにする

まず、**最初の2人がたまたま間違ったシグナル**を受け取った場合を、手で追ってみよう（正解は1だが、最初の2人のシグナルが 0 だったとする）。

In [ ]:
正解 = 1
# 運の悪い並び：最初の2人だけシグナルが間違い(0)、残りは全員正しいシグナル(1)
私的シグナル = [0, 0, 1, 1, 1, 1, 1, 1, 1, 1]

m = 0          # 公開情報：これまでに「情報を持って」選ばれた選択の差（1寄り −1寄り）
print("順番  自分のシグナル  前の人の傾き  →  実際の選択   状態")
for i, s in enumerate(私的シグナル):
    if abs(m) >= 2:                      # 既にカスケード：自分のシグナルを捨てて多数に従う
        選択 = 1 if m > 0 else 0
        状態 = "カスケード（シグナルを無視）"
    else:                                # まだ自分のシグナルが活きる
        v = m + (1 if s == 1 else -1)
        選択 = 1 if v > 0 else (0 if v < 0 else s)
        m += (1 if s == 1 else -1)
        状態 = "自分のシグナルで判断"
    印 = "✓正解" if 選択 == 正解 else "✗誤り"
    print(f" {i+1:>2}　　　{s}　　　　　{m:+d}　　　　 {選択}　 {印}　{状態}")

**3人目以降は、自分のシグナルが「正解(1)」を指しているのに、全員が間違った選択(0)をしてしまう。**

これは3人目が愚かだからではない。むしろ **合理的だからこそ** ―― 「先に2人が0を選んだ。自分のシグナル1つでは、2人ぶんの判断を覆せない」と正しく計算した結果、自分の情報を捨てて多数に従った。だが先頭2人の選択は **偶然の外れシグナル** から生まれたもので、それが後続全員に伝染した。これが **情報カスケード** だ。

> 💬 **独立が壊れる瞬間**
> 
> カスケードが始まると、3人目以降の選択は **自分の私的情報を反映しなくなる**。みんなが「前の人」という同じ情報源だけを見て動くので、判断が互いに相関し、独立でなくなる。新しい情報が集団に入らなくなるのだ。

---
## 4. シミュレーション ―― 群れは、独立な群衆に負ける

今のは運の悪い1例。では、シグナルをランダムにして何万回も試すと、**「前の人を見て真似る群れ」** と **「各自が自分のシグナルだけで独立に判断する群衆」** で、正解率はどう違うか。

どちらも一人の賢さ（シグナル正解率70%）は同じ。20人の集団で比べる。

In [ ]:
rng = np.random.default_rng(12)
q, N, 試行 = 0.7, 20, 30000
正解 = 1

群れ正解 = 0     # カスケード（前の人を見て真似る）：多数派が正解した回
独立正解 = 0     # 各自が自分のシグナルだけで判断 → 多数決
for _ in range(試行):
    シグナル = (rng.random(N) < q).astype(int)    # 各人の私的シグナル（70%で正解1）
    # --- 群れ（情報カスケード）---
    m, 選択列 = 0, []
    for s in シグナル:
        if abs(m) >= 2:
            c = 1 if m > 0 else 0
        else:
            v = m + (1 if s == 1 else -1)
            c = 1 if v > 0 else (0 if v < 0 else int(s))
            m += (1 if s == 1 else -1)
        選択列.append(c)
    群れ正解 += (np.sum(選択列) > N / 2)
    # --- 独立な群衆（各自シグナルで多数決）---
    独立正解 += (シグナル.sum() > N / 2)

print(f"一人の賢さ（シグナル正解率）はどちらも {q:.0%}、集団は {N} 人。\n")
print(f"  前の人を真似る『群れ』が正解 　： {群れ正解/試行:.1%}")
print(f"  各自が独立に判断した『群衆』が正解： {独立正解/試行:.1%}")
print(f"\n→ 群れが集団的に間違う確率は {(1-群れ正解/試行):.0%}、独立な群衆は {(1-独立正解/試行):.0%}。")
print(f"   真似ることで、集団が間違える危険が約 {(1-群れ正解/試行)/(1-独立正解/試行):.0f} 倍に増えた。")

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(["前の人を真似る\n（情報カスケード）", "各自が独立に判断\n（群衆）"],
        [群れ正解/試行, 独立正解/試行], color=["#e8503a", "#3949ab"])
plt.axhline(q, ls="--", color="gray", label=f"一人の正解率 {q:.0%}")
plt.ylim(0, 1); plt.ylabel("集団が正解した割合")
plt.title("同じ賢さの個人でも、真似ると群衆の知恵を捨ててしまう")
for i, v in enumerate([群れ正解/試行, 独立正解/試行]):
    plt.text(i, v + 0.02, f"{v:.0%}", ha="center")
plt.legend(); plt.show()

**独立に判断した群衆は約95%正解。だが前の人を真似た群れは約85%にとどまる。** 一人ひとりの賢さは同じなのに、だ。

真似ることで、後から来た18人の **私的情報が集団に入らなくなり**、最初の数人の（時に外れた）判断に全員が引きずられる。**個人は合理的でも、集団は系統的に劣化する。** これが情報カスケードの怖さだ。

現実のカスケード：株や仮想通貨の **バブル**、根拠の薄い **流行**、銀行の **取り付け騒ぎ**、レビュー★の数だけで買う行動、SNSの炎上 ―― どれも「他人が動いた」を見て自分の判断を手放した連鎖だ。

---
## 5. ディスカッション（教室で）

次の問いを、グループで話し合ってみよう。

1. あなたが最近「他人がそうしているから」を主な理由に選んだもの（店・商品・進路・意見）は？ そのとき、自分の私的な情報をどれだけ捨てただろう？
2. カスケードを **止める** には何が要るか？（ヒント：早い段階で“自分のシグナル”を口に出す人、匿名で同時に投票する仕組み、など）
3. 「みんなが賛成しているから正しい」は、どんな条件のときに信用できて、どんな条件のとき危険か。

---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| 情報カスケード | 他人の行動を見て、自分の私的情報を捨てて多数に従う連鎖 |
| なぜ起きる | 個人が**合理的**に「自分のシグナル1つでは多数を覆せない」と判断するから |
| 何が壊れる | 後続の選択が私的情報を反映せず、判断が互いに**相関**＝独立が崩れる |
| 結果 | 同じ賢さでも、独立な群衆95%に対し、真似る群れは85%。集団が系統的に誤る |

- カスケードは「個人が愚か」だから起きるのではない。**合理的な個人**でも起きる。
- 真似ることで、群衆の知恵（独立な多数の情報）が失われる。

> **課題（Moodle）**：カスケードのシミュレーション結果の解釈（自動採点）＋「独立性が壊れると集団の判断はどうなるか」の記述。詳しくはMoodleの第12回課題を見ること。

> **次回予告**：第13回「『空気を読む』ことの愚かさ：コンドルセの陪審定理」。今日の裏返し ―― **独立な多数決はほぼ確実に正解する**（コンドルセ）。だが空気を読んで判断が相関すると、その奇跡が崩れる。第1回の予告編を、いよいよ回収する。